# 01 · Data and splits (task T2)

Builds the train / validation / test splits that every experiment uses, for 5 seeds × 5 fraud rates (20%, 5%, 1%, 0.5%, natural 0.172%), and saves them to Google Drive.

The indices are saved once and reused, so both model families are always scored on exactly the same rows. A short fingerprint of all indices is printed at the end. It goes into the repo so anyone can check they rebuilt the same splits.

This notebook only needs a CPU, but running it in the same GPU session as the smoke test is fine.

In [1]:
# 1. Mount Google Drive: the split file (and later all raw results) live there,
#    so they survive Colab disconnects and stay out of git.
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# 2. Get the latest code and install the pinned libraries (same as the smoke test).
import os, sys
REPO_URL = "https://github.com/Pakkabhabad18/tabular-fm-fraud-detection.git"
REPO_DIR = "/content/tabular-fm-fraud-detection"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
!pip install -q -r {REPO_DIR}/requirements.txt
sys.path.insert(0, f"{REPO_DIR}/src")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 850.0/850.0 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.4/310.4 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.9/595.9 kB 33.9 MB/s eta 0:00:00


In [3]:
# 3. Run the unit tests first. They check the split and scoring code on fake data
#    with known answers. If anything fails here, stop: the splits would be wrong.
!cd {REPO_DIR} && PYTHONPATH=src python -m pytest tests -q

.....................                                                    [100%]
21 passed in 13.73s


In [4]:
# 4. Load ULB. load_ulb() also checks it has 284,807 rows, 492 frauds and no gaps.
from fraudicl.data import load_ulb, features_and_target
df = load_ulb()
X, y = features_and_target(df)
print(X.shape, "frauds:", y.sum())

Using Colab cache for faster access to the 'creditcardfraud' dataset.
(284807, 30) frauds: 492


In [5]:
# 5. Build all 25 splits (5 seeds x 5 fraud rates) and save them to Drive.
from fraudicl.config import DRIVE_DIR
from fraudicl.data import make_splits, save_splits, fingerprint

splits = make_splits(y)
os.makedirs(f"{DRIVE_DIR}/splits", exist_ok=True)
save_splits(splits, f"{DRIVE_DIR}/splits/ulb_splits.npz")
print(len(splits), "splits saved to", f"{DRIVE_DIR}/splits/ulb_splits.npz")
print("fingerprint:", fingerprint(splits))

25 splits saved to /content/drive/MyDrive/tabular-fm-fraud-detection/splits/ulb_splits.npz
fingerprint: 431e76c7a7e44ec0


In [6]:
# 6. Row and fraud counts for seed 0. The other seeds have the same counts,
#    only different rows.
import pandas as pd
counts = pd.DataFrame([s.summary(y) for s in splits])
cols = ["level", "train_rows", "train_frauds", "val_rows", "val_frauds", "test_rows", "test_frauds"]
counts[counts.seed == 0][cols]

,level,train_rows,train_frauds,val_rows,val_frauds,test_rows,test_frauds
0,20,1475,295,495,99,490,98
1,5,5900,295,1980,99,1960,98
2,1,29500,295,9900,99,9800,98
3,0.5,59000,295,19800,99,19600,98
4,natural,170883,295,56962,99,56962,98


**How to read this table.** Each row is one fraud rate. The fraud counts stay the same down the table (about 295 / 99 / 98), because frauds are never dropped. Only the number of legitimate rows grows as the fraud rate falls.

At 20%, training has only about 1,475 rows. So the context budgets of 2K, 5K and 10K rows can't run at that level: the training pool is smaller than the context. That gap in the grid is expected, not a bug.

In [7]:
# 7. Print everything that goes into the repo: copy this output into
#    results/ulb_split_counts.csv and results/ulb_splits_fingerprint.txt.
print("fingerprint:", fingerprint(splits))
print(counts.to_csv(index=False))

fingerprint: 431e76c7a7e44ec0
seed,level,train_rows,train_frauds,train_fraud_rate,val_rows,val_frauds,val_fraud_rate,test_rows,test_frauds,test_fraud_rate
0,20,1475,295,0.2,495,99,0.2,490,98,0.2
0,5,5900,295,0.05,1980,99,0.05,1960,98,0.05
0,1,29500,295,0.01,9900,99,0.01,9800,98,0.01
0,0.5,59000,295,0.005,19800,99,0.005,19600,98,0.005
0,natural,170883,295,0.0017263273701889597,56962,99,0.0017380007724447878,56962,98,0.0017204452090867595
1,20,1475,295,0.2,495,99,0.2,490,98,0.2
1,5,5900,295,0.05,1980,99,0.05,1960,98,0.05
1,1,29500,295,0.01,9900,99,0.01,9800,98,0.01
1,0.5,59000,295,0.005,19800,99,0.005,19600,98,0.005
1,natural,170883,295,0.0017263273701889597,56962,99,0.0017380007724447878,56962,98,0.0017204452090867595
2,20,1475,295,0.2,495,99,0.2,490,98,0.2
2,5,5900,295,0.05,1980,99,0.05,1960,98,0.05
2,1,29500,295,0.01,9900,99,0.01,9800,98,0.01
2,0.5,59000,295,0.005,19800,99,0.005,19600,98,0.005
2,natural,170883,295,0.0017263273701889597,56962,99,0.0017380007724447878,56962,98,0.0017204